# FLUX v2 — Phase 2.5: Field Evolution Generator

**Goal:** Train a FLUX-native generator that predicts the **next wave** through field physics.

```
text → CSE (frozen) → WaveChunker (frozen) → chunk_waves[:n]
  → FieldEvolutionGenerator:
      → FieldSeeder: place prefix waves into local field [B, S, 512]
      → FieldEvolver: settle field through K energy-minimization steps
      → FieldReader: extract settled state at position n+1 → wave [432]
  → WaveToText (frozen) → decoded bytes
```

This is the SPECIFICATION's core principle made real:
  **"Input → Perturbation → Field Settles → Output extracted from settled state"**

Key differences from WRU approach:
- **No mean-pooling** — full prefix sequence enters the field
- **Energy settling IS the computation** — not a separate prediction head
- **Local interactions** — 1D depthwise convolution, not global attention
- **Energy-gated updates** — high energy = fluid, low energy = stable

**Acceptance criteria:**
- Decode gate avg byte accuracy ≥ 60%, min ≥ 30%
- Different contexts → different predictions (avg pairwise cosine < 0.90)
- Energy monotonically decreases during settling
- Output always a valid wave (bounded, non-degenerate bands)

**Ordered cells:** Setup → Refresh → Hardware → Smoke Test → Training → Diagnostics
→ Upload → Test 1 → Test 2 → Test 3 → Demo 1 → Demo 2 → Save Results → Final Summary

---
> **Requires Phase 1 + Phase 2 v2 checkpoints.**
> HuggingFace: [UnseenGAP/FLUX](https://huggingface.co/UnseenGAP/FLUX)
> GitHub: [Unseengap/FLUX @ v2](https://github.com/Unseengap/FLUX/tree/v2)

## Cell 1 — SETUP & CLONE
First-time only. Installs deps, clones v2 branch.

In [ ]:
# ── Cell 1: SETUP & CLONE ─────────────────────────────────────────────────────
import subprocess, os, sys

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'numpy', 'scipy', 'datasets', 'evaluate', 'transformers',
    'huggingface_hub', 'matplotlib', 'tqdm', 'rich'
], check=True)

# Clone repo (if not already present)
REPO_PATH = '/content/FLUX' if os.path.exists('/content') else os.path.expanduser('~/FLUX')
BRANCH = 'v2'

if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '-b', BRANCH,
                   'https://github.com/Unseengap/FLUX.git', REPO_PATH], check=True)
    print(f'  ✓ Cloned FLUX repo to {REPO_PATH}')
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', 'origin', BRANCH])
    print(f'  ✓ Pulled latest from {BRANCH}')

# Install package
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_PATH, '-q'], check=True)
print('  ✓ Setup complete')

## Cell 2 — REFRESH ⟳
**Run at the start of every session and after every bug fix.**
Clears namespace, pulls latest code, wipes stale results.

In [ ]:
# ── Cell 2: REFRESH ───────────────────────────────────────────────────────────
%reset -f
import gc, os, sys, shutil, subprocess, torch

REPO_PATH = '/content/FLUX' if os.path.exists('/content') else os.path.expanduser('~/FLUX')
RESULTS_DIR = f'{REPO_PATH}/v2/V2_results/phase2_5'
LOCAL_CKPT = f'{REPO_PATH}/checkpoints/phase2_5_v2.phase.pt'

# Clear GPU/CPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Pull latest
subprocess.run(['git', '-C', REPO_PATH, 'pull', 'origin', 'v2'], capture_output=True)

# Wipe stale results
if os.path.exists(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add paths
for p in [REPO_PATH, f'{REPO_PATH}/v2/phase1', f'{REPO_PATH}/v2/phase2', f'{REPO_PATH}/v2/phase2_5']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Resolve tokens
HF_TOKEN = os.environ.get('HF_TOKEN', '')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get('HF_TOKEN', '')
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN', '')
except Exception:
    pass
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    HF_TOKEN = HF_TOKEN or _secrets.get_secret('HF_TOKEN')
    GITHUB_TOKEN = GITHUB_TOKEN or _secrets.get_secret('GITHUB_TOKEN')
except Exception:
    pass
HF_TOKEN = HF_TOKEN.strip() if HF_TOKEN else ''
GITHUB_TOKEN = GITHUB_TOKEN.strip() if GITHUB_TOKEN else ''

print(f'  REPO_PATH   : {REPO_PATH}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')
print(f'  HF_TOKEN    : {"set" if HF_TOKEN else "NOT SET"}')
print(f'  GITHUB_TOKEN: {"set" if GITHUB_TOKEN else "NOT SET"}')
print('  ✓ Refresh complete')

## Cell 3 — HARDWARE & CREDENTIALS

In [ ]:
# ── Cell 3: HARDWARE & CREDENTIALS ────────────────────────────────────────────
import sys, os, shutil
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import torch
from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=2)
log.cell_start('Cell 3 — Hardware & Credentials')

DEVICE = get_device()
log.info(f'Device: {DEVICE}')

if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_mem / 1e9
    log.info(f'GPU: {props.name} ({vram_gb:.1f} GB VRAM)')
    if vram_gb < 8:
        log.warning('< 8GB VRAM — may need smaller batch size')
elif DEVICE == 'mps':
    log.info('Apple Silicon MPS')

# Download prior checkpoints if needed
ckpt_dir = f'{REPO_PATH}/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

for fname in ['phase1_v2.phase.pt', 'phase2_v2.phase.pt']:
    local = os.path.join(ckpt_dir, fname)
    if not os.path.exists(local):
        log.info(f'Downloading {fname} from HuggingFace...')
        try:
            from huggingface_hub import hf_hub_download
            dl = hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename=f'v2/{fname}',
                token=HF_TOKEN,
                local_dir=ckpt_dir,
            )
            dl_real = os.path.realpath(dl)
            if dl_real != local:
                shutil.copy2(dl_real, local)
            log.success(f'{fname} downloaded')
        except Exception as e:
            log.error(f'Failed to download {fname}: {e}')
    else:
        log.info(f'{fname} already present')

log.cell_end('Cell 3 — Hardware & Credentials', 'PASS')

## Cell 4 — SMOKE TEST
Verifies all Phase 2.5 imports work and a minimal FieldEvolutionGenerator forward pass
produces correct shapes. Also verifies Phase 1 + Phase 2 codecs load.

In [ ]:
# ── Cell 4: SMOKE TEST ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

log.cell_start('Cell 4 — Smoke Test (Phase 2.5 Field Evolution)')
_all_pass = True

# ── Import checks ─────────────────────────────────────────────────────────────
_imports = [
    ('cse',                        'ContinuousSemanticEncoder'),
    ('wave_chunker',               'WaveChunker'),
    ('wave_to_text',               'WaveToText'),
    ('decode_gate',                'byte_accuracy'),
    ('train_codec',                'load_phase1_checkpoint'),
    ('wave_to_field',              'WaveToField'),
    ('field_to_wave',              'FieldToWave'),
    ('field_evolution_generator',  'FieldEvolutionGenerator'),
    ('field_evolution_generator',  'BAND_SLICES'),
    ('train_field_evolution',      'train_field_evolution'),
    ('train_field_evolution',      'load_phase1_and_phase2'),
    ('train_field_evolution',      'sample_batch'),
]

for module_name, symbol_name in _imports:
    try:
        _mod = __import__(module_name, fromlist=[symbol_name])
        getattr(_mod, symbol_name)
        print(f'  ✓ import {module_name}.{symbol_name}')
    except Exception as e:
        print(f'  ✗ import {module_name}.{symbol_name}  →  {e}')
        _all_pass = False

# ── Forward pass: FieldEvolutionGenerator with random input ────────────────────
import torch
from field_evolution_generator import FieldEvolutionGenerator

try:
    _gen = FieldEvolutionGenerator()
    _prefix = torch.randn(4, 8, 432)  # [B=4, N=8, wave_dim=432]
    _pred, _info = _gen(_prefix)
    assert _pred.shape == (4, 432), f'Expected (4, 432), got {_pred.shape}'
    assert 'energy_trace' in _info
    assert 'energy_drop' in _info
    _params = _gen.count_parameters()
    print(f'  ✓ Generator forward: input [4, 8, 432] → output {list(_pred.shape)}')
    print(f'  ✓ Generator parameters: {_params:,}')
    _summary = _gen.parameter_summary()
    for comp, cnt in _summary.items():
        if comp != 'total':
            print(f'    {comp}: {cnt:,}')
except Exception as e:
    print(f'  ✗ Generator forward failed: {e}')
    _all_pass = False

# ── Energy settling check ──────────────────────────────────────────────────────
try:
    _trace = _info['energy_trace'][0]  # [settle_steps+1]
    _drop = _trace[0] - _trace[-1]
    print(f'  ✓ Energy trace: {" → ".join(f"{e:.2f}" for e in _trace.tolist())}')
    print(f'  ✓ Energy drop: {_drop:.4f}')
except Exception as e:
    print(f'  ✗ Energy trace check failed: {e}')
    _all_pass = False

# ── Output boundedness (tanh) ──────────────────────────────────────────────────
try:
    _max_abs = _pred.abs().max().item()
    assert _max_abs < 2.0, f'Max abs {_max_abs} ≥ 2.0'
    print(f'  ✓ Output bounded: max_abs={_max_abs:.4f} (tanh → [-1,1])')
except Exception as e:
    print(f'  ✗ Output boundedness failed: {e}')
    _all_pass = False

# ── Phase 1 + Phase 2 checkpoint loading ──────────────────────────────────────
_components = None
try:
    from train_field_evolution import load_phase1_and_phase2
    _components = load_phase1_and_phase2(device='cpu', hf_token=HF_TOKEN)
    print(f'  ✓ Phase 1 + Phase 2 loaded: {list(_components.keys())}')
except Exception as e:
    print(f'  ✗ Phase 1+2 loading failed: {e}')
    _all_pass = False

# ── Quick end-to-end check ─────────────────────────────────────────────────────
if _components is not None:
    try:
        _cse = _components['cse']
        _chunker = _components['chunker']
        _test_text = 'The cat sat on the mat'
        _wave = _cse.encode(_test_text)
        _pairs = _chunker.chunk_with_bytes(_wave.full, _test_text.encode('utf-8'))
        assert len(_pairs) >= 3, f'Need ≥3 chunks, got {len(_pairs)}'
        _cw = torch.stack([p[0] for p in _pairs])
        _prefix_e2e = _cw[:3].unsqueeze(0)  # [1, 3, 432]
        _pred_test, _info_test = _gen(_prefix_e2e)
        print(f'  ✓ End-to-end: \'{_test_text}\' → {len(_pairs)} chunks → prefix[3] → Generator → {list(_pred_test.shape)}')
    except Exception as e:
        print(f'  ✗ End-to-end check failed: {e}')
        _all_pass = False
else:
    print(f'  ⚠ End-to-end check skipped — Phase 1+2 loading failed')

if _all_pass:
    log.success('SMOKE TEST PASSED — all imports and shapes OK')
    log.cell_end('Cell 4 — Smoke Test', 'PASS')
else:
    log.error('SMOKE TEST FAILED — fix errors before training')
    log.cell_end('Cell 4 — Smoke Test', 'FAIL')
    raise RuntimeError('Smoke test failed — do not proceed to training.')

## Cell 5 — TRAINING
Trains the **FieldEvolutionGenerator** for next-wave prediction through field physics.
Phase 1 (CSE, WaveChunker, WTT) and Phase 2 (WaveToField, FieldToWave) are **FROZEN**.
Default: **30,000 steps** (~30 min on T4/L4). Full prefix enters the field — no mean-pooling.

In [ ]:
# ── Cell 5: TRAINING ──────────────────────────────────────────────────────────
import sys, os, importlib
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import train_field_evolution as _tfe_module
importlib.reload(_tfe_module)
from train_field_evolution import train_field_evolution

import torch
from flux_utils import PhaseLogger

log.cell_start('Cell 5 — Training (Phase 2.5 Field Evolution)')

# ── Hyperparameters ───────────────────────────────────────────────────────────
TRAIN_STEPS       = 30_000
BATCH_SIZE        = 32
LEARNING_RATE     = 3e-4
LOG_EVERY         = 500
SAVE_EVERY        = 5_000
GATE_CHECK_EVERY  = 2_500
COSINE_WEIGHT     = 1.0    # Direction alignment
DECODE_WEIGHT     = 5.0    # TEXT fidelity is the ACTUAL objective
ENERGY_WEIGHT     = 0.1    # Encourage energy decrease during settling
SETTLE_STEPS      = 4      # Energy-settling iterations per forward pass
MAX_PREFIX_LEN    = 32     # Max prefix length to use

log.info(f'Steps            : {TRAIN_STEPS:,}')
log.info(f'Batch size       : {BATCH_SIZE}')
log.info(f'Learning rate    : {LEARNING_RATE}')
log.info(f'Device           : {DEVICE}')
log.info(f'cosine_weight    : {COSINE_WEIGHT}')
log.info(f'decode_weight    : {DECODE_WEIGHT}')
log.info(f'energy_weight    : {ENERGY_WEIGHT}')
log.info(f'settle_steps     : {SETTLE_STEPS}')
log.info(f'max_prefix_len   : {MAX_PREFIX_LEN}')
log.separator('Starting Phase 2.5 Field Evolution training...')

# ── Run training ──────────────────────────────────────────────────────────────
_trained_gen = train_field_evolution(
    steps            = TRAIN_STEPS,
    batch_size       = BATCH_SIZE,
    device           = DEVICE,
    lr               = LEARNING_RATE,
    log_every        = LOG_EVERY,
    save_every       = SAVE_EVERY,
    gate_check_every = GATE_CHECK_EVERY,
    cosine_weight    = COSINE_WEIGHT,
    decode_weight    = DECODE_WEIGHT,
    energy_weight    = ENERGY_WEIGHT,
    settle_steps     = SETTLE_STEPS,
    max_prefix_len   = MAX_PREFIX_LEN,
    upload_hf        = False,
    hf_token         = HF_TOKEN,
)

log.success('Training complete')
log.metric('Checkpoint', LOCAL_CKPT)
log.cell_end('Cell 5 — Training (Phase 2.5)', 'PASS')

## Cell 6 — TRAINING DIAGNOSTICS
**Run immediately after training.** Checks all show-stoppers before uploading.

In [ ]:
# ── Cell 6: TRAINING DIAGNOSTICS ──────────────────────────────────────────────
import json, math, torch, os, sys
import torch.nn.functional as F
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

log.cell_start('Cell 6 — Training Diagnostics')
_diag_pass = True

# Load checkpoint
_ckpt = torch.load(LOCAL_CKPT, map_location='cpu', weights_only=False)
print(f'  Checkpoint keys: {list(_ckpt.keys())}')
print(f'  Phase: {_ckpt["phase"]}')
print(f'  Timestamp: {_ckpt["timestamp"]}')

_metrics = _ckpt.get('metrics', {})
print(f'  Metrics: {json.dumps({k: f"{v:.4f}" if isinstance(v, float) else str(v) for k, v in _metrics.items()}, indent=2)}')

# Check decode gate result
_gate_avg = _metrics.get('gate_avg', _metrics.get('best_gate_avg', 0.0))
_gate_min = _metrics.get('gate_min', 0.0)
_gate_passed = _metrics.get('gate_passed', False)

print(f'\n  Decode gate: avg={_gate_avg:.1%}  min={_gate_min:.1%}  passed={_gate_passed}')

# Check config
_config = _ckpt.get('config', {})
print(f'  Config: {json.dumps(_config, indent=2)}')

# Check model state dict size
_sd = _ckpt.get('state_dict', {})
if 'field_evolution_generator' in _sd:
    _n_tensors = len(_sd['field_evolution_generator'])
    _n_params = sum(v.numel() for v in _sd['field_evolution_generator'].values())
    print(f'  FieldEvolutionGenerator: {_n_tensors} tensors, {_n_params:,} parameters')
else:
    print(f'  ✗ state_dict missing field_evolution_generator key!')
    _diag_pass = False

# Quick forward pass with loaded weights
from field_evolution_generator import FieldEvolutionGenerator
_gen_diag = FieldEvolutionGenerator(
    settle_steps=_config.get('settle_steps', 4),
    kernel_size=_config.get('kernel_size', 5),
)
_gen_diag.load_state_dict(_sd['field_evolution_generator'])
_gen_diag.eval()

_test_input = torch.randn(2, 5, 432)
_test_pred, _test_info = _gen_diag(_test_input)
print(f'  ✓ Forward pass with loaded weights: {list(_test_pred.shape)}')
print(f'  Energy drop: {_test_info["energy_drop"].mean().item():.4f}')

if _diag_pass:
    log.success('DIAGNOSTICS PASSED')
    log.cell_end('Cell 6 — Diagnostics', 'PASS')
else:
    log.error('DIAGNOSTICS FAILED — do not upload')
    log.cell_end('Cell 6 — Diagnostics', 'FAIL')

## Cell 7 — UPLOAD TO HUGGINGFACE

In [ ]:
# ── Cell 7: UPLOAD TO HUGGINGFACE ─────────────────────────────────────────────
log.cell_start('Cell 7 — Upload to HuggingFace')

try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=LOCAL_CKPT,
        path_in_repo='v2/phase2_5_v2.phase.pt',
        repo_id='UnseenGAP/FLUX',
        token=HF_TOKEN,
    )
    _mb = os.path.getsize(LOCAL_CKPT) / 1e6
    log.success(f'Uploaded phase2_5_v2.phase.pt ({_mb:.1f} MB)')
    log.cell_end('Cell 7 — Upload', 'PASS')
except Exception as e:
    log.error(f'Upload failed: {e}')
    log.cell_end('Cell 7 — Upload', 'FAIL')

## Cell 8 — TEST 1: Predicted waves decode to readable text

In [ ]:
# ── Cell 8: TEST 1 — Decode Gate ──────────────────────────────────────────────
import importlib, sys
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import test_phase2_5_test1_fe as _t1
importlib.reload(_t1)

log.cell_start('Cell 8 — Test 1')
try:
    _t1.main()
    log.success('TEST 1 PASSED')
    log.cell_end('Cell 8 — Test 1', 'PASS')
except AssertionError as e:
    log.error(f'TEST 1 FAILED: {e}')
    log.cell_end('Cell 8 — Test 1', 'FAIL')

## Cell 9 — TEST 2: Different contexts → different predictions

In [ ]:
# ── Cell 9: TEST 2 — Diversity ────────────────────────────────────────────────
import importlib, sys
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import test_phase2_5_test2_fe as _t2
importlib.reload(_t2)

log.cell_start('Cell 9 — Test 2')
try:
    _t2.main()
    log.success('TEST 2 PASSED')
    log.cell_end('Cell 9 — Test 2', 'PASS')
except AssertionError as e:
    log.error(f'TEST 2 FAILED: {e}')
    log.cell_end('Cell 9 — Test 2', 'FAIL')

## Cell 10 — TEST 3: Output is always a valid wave

In [ ]:
# ── Cell 10: TEST 3 — Wave Validity ───────────────────────────────────────────
import importlib, sys
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import test_phase2_5_test3_fe as _t3
importlib.reload(_t3)

log.cell_start('Cell 10 — Test 3')
try:
    _t3.main()
    log.success('TEST 3 PASSED')
    log.cell_end('Cell 10 — Test 3', 'PASS')
except AssertionError as e:
    log.error(f'TEST 3 FAILED: {e}')
    log.cell_end('Cell 10 — Test 3', 'FAIL')

## Cell 11 — DEMO 1: Prompt → Field Evolution → Decoded Text

In [ ]:
# ── Cell 11: DEMO 1 ───────────────────────────────────────────────────────────
import importlib, sys
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import demo_phase2_5_demo1_fe as _d1
importlib.reload(_d1)

log.cell_start('Cell 11 — Demo 1')
_d1.main()
log.cell_end('Cell 11 — Demo 1', 'PASS')

## Cell 12 — DEMO 2: Field Evolution Energy Landscape

In [ ]:
# ── Cell 12: DEMO 2 ───────────────────────────────────────────────────────────
import importlib, sys
sys.path.insert(0, f'{REPO_PATH}/v2/phase2_5')

import demo_phase2_5_demo2_fe as _d2
importlib.reload(_d2)

log.cell_start('Cell 12 — Demo 2')
_d2.main()
log.cell_end('Cell 12 — Demo 2', 'PASS')

## Cell 13 — SAVE RESULTS

In [ ]:
# ── Cell 13: SAVE RESULTS ─────────────────────────────────────────────────────
import shutil, subprocess, os

log.cell_start('Cell 13 — Save Results')

# Copy logs
_log_src = f'{REPO_PATH}/logs/phase2.log'
if os.path.exists(_log_src):
    shutil.copy2(_log_src, f'{RESULTS_DIR}/phase2_5.log')
    print(f'  ✓ Copied log to {RESULTS_DIR}')

# Copy results MD
_results_src = f'{REPO_PATH}/results/RESULTS_PHASE_2.md'
if os.path.exists(_results_src):
    shutil.copy2(_results_src, f'{RESULTS_DIR}/RESULTS_PHASE_2_5.md')

# Push to GitHub
try:
    _git = lambda cmd: subprocess.run(
        ['git', '-C', REPO_PATH] + cmd.split(),
        capture_output=True, text=True
    )
    _git('add v2/V2_results/')
    _git('commit -m Phase_2.5_field_evolution_results')
    if GITHUB_TOKEN:
        _remote = f'https://{GITHUB_TOKEN}@github.com/Unseengap/FLUX.git'
        _git(f'push {_remote} v2')
        log.success('Results pushed to GitHub')
    else:
        log.warning('No GITHUB_TOKEN — results saved locally only')
except Exception as e:
    log.warning(f'Git push failed: {e}')

# Upload logs to HuggingFace
try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    if os.path.exists(_log_src):
        api.upload_file(
            path_or_fileobj=_log_src,
            path_in_repo='v2/logs/phase2_5.log',
            repo_id='UnseenGAP/FLUX',
            token=HF_TOKEN,
        )
        log.success('Log uploaded to HuggingFace')
except Exception as e:
    log.warning(f'HF log upload failed: {e}')

log.cell_end('Cell 13 — Save Results', 'PASS')

## Cell 14 — FINAL SUMMARY

### Phase 2.5 v2: Field Evolution Generator

**Architecture:** FieldEvolutionGenerator
- **FieldSeeder:** Places prefix waves into a compact local field with learned position embeddings
- **FieldEvolver:** Differentiable energy settling via depthwise-separable 1D convolution + energy gates
- **FieldReader:** Extracts prediction at position n+1 via local contextual attention → wave projection

**Physics principles realized:**
- Energy settling IS the computation (SPECIFICATION.md canonical design)
- Local interactions only (kernel_size=5, not global attention)
- High energy = fluid (changes easily), Low energy = stable (resists change)
- Energy monotonically decreases during settling
- Full prefix sequence enters the field (no lossy mean-pooling)

**Training objective:**
- Cosine similarity loss (direction alignment)
- Decode loss (WTT cross-entropy = text fidelity)
- Energy loss (encourage energy decrease during settling)

**Acceptance criteria:**
- [ ] Decode gate avg ≥ 60%, min ≥ 30%
- [ ] Different contexts → different predictions (avg cosine < 0.90)
- [ ] Energy monotonically decreasing
- [ ] Output always a valid wave